In [ ]:
import json
from pathlib import Path

In [ ]:
data_path = Path("../data/raw/exploring_set.jsonl")
data_path

In [ ]:
data_path.exists()

In [ ]:
with data_path.open("r",encoding="utf-8") as file:
    first_line = file.readline()

first_line[:300]

In [ ]:
first_record = json.loads(first_line)
type(first_record)

In [ ]:
first_record.keys()

In [ ]:
for key,value in first_record.items():
    print(key,"->",type(value).__name__,"示例：",str(value)[:80])

In [ ]:
with data_path.open("r", encoding="utf-8") as file:
    total_rows = sum(1 for line in file)

total_rows

## 查重

In [ ]:
note_ids = set()

with data_path.open("r",encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)
        note_ids.add(record["note_id"])

unique_notes = len(note_ids)
duplicate_rows = total_rows - unique_notes

unique_notes,duplicate_rows

In [ ]:
duplicate_rate = duplicate_rows / total_rows
f"{duplicate_rate:.2%}"

In [ ]:
from collections import Counter
note_id_counts = Counter()
with data_path.open("r",encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)
        note_id_counts[record["note_id"]] += 1
note_id_counts.most_common(10)

In [ ]:
target_note_id = "67b9f711000000000e00499c"
target_records = []

with data_path.open("r",encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)

        if record["note_id"] == target_note_id:
            target_records.append(record)

len(target_records)

In [ ]:
for index,record in enumerate(target_records,start=1):
    print(
        index,
        "时间：",record.get("local_time"),
        "点赞：",record.get("liked_count"),
        "收藏：",record.get("collected_count"),
        "评论：",record.get("comments_count"),
        "领域：",record.get("domain")
    )

In [ ]:
fields_to_check = [
    "note_title",
    "time",
    "local_time",
    "desc",
    "user_id"
]

for field in fields_to_check:
    unique_values = {record.get(field) for record in target_records}
    print(field, "不同值数量：", len(unique_values))

In [ ]:
schema_counts = Counter()

with data_path.open("r", encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)
        schema = tuple(sorted(record.keys()))
        schema_counts[schema] += 1

len(schema_counts)

## 检查整个文件共有多少种字段结构

In [ ]:
for schema_number, (schema, count) in enumerate(
    schema_counts.most_common(),
    start=1
):
    print("结构编号：", schema_number)
    print("记录数量：", count)
    print("字段列表：", schema)
    print()

In [ ]:
field_mapping = {
    "likes": "liked_count",
    "collections": "collected_count",
    "comments": "comments_count",
    "note_content": "desc"
}

field_mapping

In [ ]:
with data_path.open("r", encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)

        if "likes" in record:
            schema1_example = record
            break

schema1_example.keys()

In [ ]:
normalized_example = schema1_example.copy()

for old_name, new_name in field_mapping.items():
    normalized_example[new_name] = normalized_example.pop(old_name)

normalized_example.keys()

In [ ]:
for old_name, new_name in field_mapping.items():
    values_equal = schema1_example[old_name] == normalized_example[new_name]

    print(
        old_name,
        "→",
        new_name,
        "值是否一致：",
        values_equal
    )

## 字段改名函数

In [ ]:
def normalize_field_names(record):
    normalized_record = record.copy()

    for old_name, new_name in field_mapping.items():
        if old_name in normalized_record:
            normalized_record[new_name] = normalized_record.pop(old_name)

    return normalized_record

In [ ]:
test_schema1 = normalize_field_names(schema1_example)
test_schema2 = normalize_field_names(first_record)

print("结构1处理后的字段：", test_schema1.keys())
print("结构2是否保持不变：", test_schema2 == first_record)

In [ ]:
standard_fields = [
    "note_id",
    "note_title",
    "time",
    "local_time",
    "desc",
    "liked_count",
    "collected_count",
    "comments_count",
    "domain",
    "user_id"
]

standard_fields

In [ ]:
def normalize_record(record):
    normalized_record = record.copy()

    for old_name, new_name in field_mapping.items():
        if old_name in normalized_record:
            normalized_record[new_name] = normalized_record.pop(old_name)

    for field in standard_fields:
        if field not in normalized_record:
            normalized_record[field] = None

    return normalized_record

In [ ]:
test_records = [
    schema1_example,
    first_record,
    target_records[0]
]

for schema_number, record in enumerate(test_records, start=1):
    normalized_record = normalize_record(record)

    fields_are_standard = (
        set(normalized_record.keys()) == set(standard_fields)
    )

    print(
        "结构", schema_number,
        "字段是否统一：", fields_are_standard,
        "domain：", normalized_record["domain"],
        "user_id：", normalized_record["user_id"]
    )

## 把全部 91,517 条原始记录标准化，并暂存在 Notebook 内存中

In [ ]:
normalized_records = []

with data_path.open("r", encoding="utf-8") as file:
    for line in file:
        original_record = json.loads(line)
        normalized_record = normalize_record(original_record)
        normalized_records.append(normalized_record)

len(normalized_records)

## 把标准化后的列表转换成 Pandas 数据表

In [ ]:
import pandas as pd

df = pd.DataFrame(
    normalized_records,
    columns=standard_fields
)

df.shape

In [ ]:
df.dtypes

## 为每条记录计算互动总量

In [ ]:
df["total_interactions"] = (
    df["liked_count"]
    +df["collected_count"]
    +df["comments_count"]
)

df[ 
    ["liked_count",
        "collected_count",
        "comments_count",
        "total_interactions"
    ]
].head()
        

In [ ]:
df[
    [
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions"
    ]
].agg(["min", "max"])

In [ ]:
top_10_interactions = df.nlargest(
    10,
    "total_interactions"
)

top_10_interactions[
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "domain"
    ]
]

In [ ]:
suspicious_note_id = "6777c495000000000900eecc"

df.loc[
    df["note_id"] == suspicious_note_id,
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "local_time",
        "domain"
    ]
]

In [ ]:
with data_path.open("r", encoding="utf-8") as file:
    for line in file:
        raw_record = json.loads(line)

        if raw_record["note_id"] == suspicious_note_id:
            suspicious_raw_record = raw_record
            break

for field in [
    "liked_count",
    "collected_count",
    "comments_count"
]:
    value = suspicious_raw_record.get(field)

    print(
        field,
        "原始值：", value,
        "数据类型：", type(value).__name__
    )

In [ ]:
interaction_columns = [
    "liked_count",
    "collected_count",
    "comments_count",
    "total_interactions"
]

df[interaction_columns].quantile(
    [0.5, 0.9, 0.95, 0.99, 0.999, 1.0]
)

In [ ]:
extreme_collection = df["collected_count"] > 1_000_000
extreme_comments = df["comments_count"] > 1_000_000
extreme_any = extreme_collection | extreme_comments

print("收藏超过100万的记录数：", extreme_collection.sum())
print("评论超过100万的记录数：", extreme_comments.sum())
print("至少一项超过100万的记录数：", extreme_any.sum())

print(
    "涉及的不同note_id数量：",
    df.loc[extreme_any, "note_id"].nunique()
)

In [ ]:
extreme_records = df.loc[extreme_any]

extreme_records[
    [
        "domain",
        "user_id",
        "time"
    ]
].isna().sum()

In [ ]:
extreme_records = extreme_records.copy()

extreme_records["domain_missing"] = extreme_records["domain"].isna()
extreme_records["user_id_missing"] = extreme_records["user_id"].isna()
extreme_records["time_missing"] = extreme_records["time"].isna()

extreme_records.groupby(
    [
        "domain_missing",
        "user_id_missing",
        "time_missing"
    ]
).size()

In [ ]:
non_schema3_extremes = extreme_records.loc[
    extreme_records["domain"].notna()
    | extreme_records["user_id"].notna()
    | extreme_records["time"].isna(),
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "domain",
        "time"
    ]
]

non_schema3_extremes

In [ ]:
positive_like_count = df["liked_count"].where(
    df["liked_count"] > 0
)

df["collection_like_ratio"] = (
    df["collected_count"] / positive_like_count
)

df["comment_like_ratio"] = (
    df["comments_count"] / positive_like_count
)

comparison_note_ids = [
    "6777c495000000000900eecc",
    "678ce103000000001803c791",
    "6836a12200000000120050d2"
]

df.loc[
    df["note_id"].isin(comparison_note_ids),
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "collection_like_ratio",
        "comment_like_ratio"
    ]
]

In [ ]:
df[
    [
        "collection_like_ratio",
        "comment_like_ratio"
    ]
].quantile(
    [0.5, 0.9, 0.95, 0.99, 0.999, 1.0]
)

In [ ]:
candidate_anomaly = (
    (
        (df["collected_count"] > 1_000_000)
        & (df["collection_like_ratio"] > 10)
    )
    |
    (
        (df["comments_count"] > 1_000_000)
        & (df["comment_like_ratio"] > 10)
    )
)

print("候选异常记录数：", candidate_anomaly.sum())

print(
    "涉及的不同note_id数量：",
    df.loc[candidate_anomaly, "note_id"].nunique()
)

In [ ]:
candidate_anomaly_records = df.loc[candidate_anomaly].copy()

candidate_anomaly_records.nsmallest(
    10,
    "total_interactions"
)[
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "collection_like_ratio",
        "comment_like_ratio",
        "domain"
    ]
]

In [ ]:
candidate_anomaly_records[
    [
        "domain",
        "user_id",
        "time"
    ]
].isna().sum()

In [ ]:
candidate_anomaly_ids = df.loc[
    candidate_anomaly,
    "note_id"
].unique()

normal_alternatives = df.loc[
    df["note_id"].isin(candidate_anomaly_ids)
    & ~candidate_anomaly
]

print("非异常替代记录数：", len(normal_alternatives))

print(
    "拥有非异常替代版本的note_id数：",
    normal_alternatives["note_id"].nunique()
)

## 先排除错误版本，再进行重复 ID 去重。

In [ ]:
df["candidate_anomaly"] = candidate_anomaly

rescuable_note_id = normal_alternatives[
    "note_id"
].iloc[0]

df.loc[
    df["note_id"] == rescuable_note_id,
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "candidate_anomaly",
        "domain",
        "time"
    ]
].sort_values(
    "candidate_anomaly"
)

In [ ]:
normal_alternatives.groupby(
    "note_id"
)[
    [
        "liked_count",
        "collected_count",
        "comments_count"
    ]
].nunique()

In [ ]:
varying_normal_ids = [
    "677cee0f000000002001f1fd",
    "67c070f8000000029029139"
]

normal_alternatives.loc[
    normal_alternatives["note_id"].isin(varying_normal_ids),
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "local_time",
        "domain",
        "time"
    ]
].sort_values(
    [
        "note_id",
        "total_interactions"
    ]
)

In [ ]:
normal_alternatives.loc[
    normal_alternatives["note_id"].str.startswith(
        "67c070f8"
    ),
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "local_time",
        "domain",
        "time"
    ]
].sort_values(
    "total_interactions"
)

## 创建一张暂时排除候选异常版本的数据表

In [ ]:
valid_versions = df.loc[
    ~df["candidate_anomaly"]
].copy()

print("排除异常后的记录数：", len(valid_versions))

print(
    "排除异常后的不同note_id数：",
    valid_versions["note_id"].nunique()
)

In [ ]:
metadata_fields = [
    "time",
    "domain",
    "user_id"
]

valid_versions["metadata_completeness"] = (
    valid_versions[metadata_fields]
    .notna()
    .sum(axis=1)
)

valid_versions[
    "metadata_completeness"
].value_counts().sort_index()

In [ ]:
sorted_valid_versions = valid_versions.sort_values(
    by=[
        "note_id",
        "total_interactions",
        "metadata_completeness"
    ],
    ascending=[
        True,
        False,
        False
    ],
    kind="mergesort"
)

In [ ]:
sorted_valid_versions.loc[
    sorted_valid_versions["note_id"].str.startswith(
        "677cee0f"
    ),
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "metadata_completeness"
    ]
]

## 正式生成去重后的数据表

In [ ]:
deduplicated_df = sorted_valid_versions.drop_duplicates(
    subset="note_id",
    keep="first"
).copy()

print("去重后的记录数：", len(deduplicated_df))

print(
    "仍然重复的note_id数量：",
    deduplicated_df.duplicated(
        subset="note_id"
    ).sum()
)

In [ ]:
rescued_id = deduplicated_df[
    "note_id"
].str.startswith(
    "678608c4"
)

unrecoverable_id = deduplicated_df[
    "note_id"
].str.startswith(
    "6777c495"
)

deduplicated_df.loc[
    rescued_id | unrecoverable_id,
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "candidate_anomaly"
    ]
]

## 创建主分析表

In [ ]:
analysis_columns = standard_fields + [
    "total_interactions"
]

clean_df = deduplicated_df[
    analysis_columns
].copy()

clean_df.shape

## 验算

In [ ]:
rescuable_ids = set(
    normal_alternatives["note_id"].unique()
)

anomaly_only_ids = (
    set(candidate_anomaly_ids)
    - rescuable_ids
)

clean_ids = set(
    clean_df["note_id"]
)

print("可挽救ID总数：", len(rescuable_ids))

print(
    "成功保留的可挽救ID数：",
    len(rescuable_ids & clean_ids)
)

print("纯异常ID总数：", len(anomaly_only_ids))

print(
    "误入主分析表的纯异常ID数：",
    len(anomaly_only_ids & clean_ids)
)

In [ ]:
max_valid_interactions = valid_versions.groupby(
    "note_id"
)["total_interactions"].max()

selected_interactions = clean_df.set_index(
    "note_id"
)["total_interactions"]

interaction_mismatch_count = (
    selected_interactions
    != max_valid_interactions
).sum()

interaction_mismatch_count

In [ ]:
valid_versions["max_interactions_for_id"] = (
    valid_versions.groupby(
        "note_id"
    )["total_interactions"].transform("max")
)

highest_interaction_versions = valid_versions.loc[
    valid_versions["total_interactions"]
    ==
    valid_versions["max_interactions_for_id"]
].copy()

print(
    "最高互动候选记录数：",
    len(highest_interaction_versions)
)

print(
    "对应的不同note_id数：",
    highest_interaction_versions["note_id"].nunique()
)

In [ ]:
highest_interaction_versions[
    "best_metadata_for_id"
] = (
    highest_interaction_versions.groupby(
        "note_id"
    )["metadata_completeness"].transform("max")
)

best_candidates = highest_interaction_versions.loc[
    highest_interaction_versions["metadata_completeness"]
    ==
    highest_interaction_versions["best_metadata_for_id"]
].copy()

print(
    "最佳候选记录数：",
    len(best_candidates)
)

print(
    "对应的不同note_id数：",
    best_candidates["note_id"].nunique()
)

In [ ]:
selected_rows_are_best = (
    deduplicated_df.index.isin(
        best_candidates.index
    )
)

wrong_selection_count = (
    ~selected_rows_are_best
).sum()

wrong_selection_count

## 检查 11 列的缺失情况

In [ ]:
missing_counts = clean_df.isna().sum()

missing_counts

In [ ]:
missing_percent = (
    missing_counts
    / len(clean_df)
    * 100
).round(2)

missing_summary = pd.DataFrame(
    {
        "missing_count": missing_counts,
        "missing_percent": missing_percent
    }
)

missing_summary

In [ ]:
clean_df["publish_datetime"] = pd.to_datetime(
    clean_df["local_time"].astype(str),
    format="%Y%m%d%H",
    errors="coerce"
)

print(
    "无法转换的发布时间数量：",
    clean_df["publish_datetime"].isna().sum()
)

clean_df[
    [
        "local_time",
        "publish_datetime"
    ]
].head()

In [ ]:
clean_df[
    "publish_datetime"
].agg(
    [
        "min",
        "max"
    ]
)

In [ ]:
clean_df["publish_datetime_from_unix"] = (
    pd.to_datetime(
        clean_df["time"],
        unit="s",
        utc=True
    )
    .dt.tz_convert("Asia/Shanghai")
    .dt.tz_localize(None)
)

time_available = clean_df["time"].notna()

time_mismatch_count = (
    clean_df.loc[
        time_available,
        "publish_datetime_from_unix"
    ].dt.floor("h")
    !=
    clean_df.loc[
        time_available,
        "publish_datetime"
    ]
).sum()

print(
    "可以交叉验证的记录数：",
    time_available.sum()
)

print(
    "两种发布时间不一致的记录数：",
    time_mismatch_count
)

In [ ]:
clean_df["domain_analysis"] = (
    clean_df["domain"].fillna("未知")
)

clean_df[
    "domain_analysis"
].value_counts()

## 所有笔记都可进入笔记级分析，但作者级分析最多只能使用 33,219 篇

In [ ]:
clean_df["has_user_id"] = (
    clean_df["user_id"].notna()
)

clean_df[
    "has_user_id"
].value_counts()

In [ ]:
clean_df["publish_year"] = (
    clean_df["publish_datetime"].dt.year
)

clean_df[
    "publish_year"
].value_counts().sort_index()

In [ ]:
clean_df["publish_year_month"] = (
    clean_df["publish_datetime"].dt.to_period("M")
)

monthly_counts = (
    clean_df["publish_year_month"]
    .value_counts()
    .sort_index()
)

monthly_counts

In [ ]:
max_value_comparison = pd.DataFrame(
    {
        "清洗前最大值": df[
            interaction_columns
        ].max(),

        "清洗后最大值": clean_df[
            interaction_columns
        ].max()
    }
)

max_value_comparison

In [ ]:
deduplicated_df.nlargest(
    10,
    "comments_count"
)[
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "collection_like_ratio",
        "comment_like_ratio",
        "domain"
    ]
]

In [ ]:
remaining_suspicious_id = (
    deduplicated_df.nlargest(
        1,
        "comments_count"
    )["note_id"].iloc[0]
)

print(
    "自动取得的note_id：",
    remaining_suspicious_id
)

remaining_suspicious = df.loc[
    df["note_id"] == remaining_suspicious_id,
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "collection_like_ratio",
        "comment_like_ratio",
        "candidate_anomaly",
        "domain",
        "time"
    ]
].sort_values(
    "total_interactions"
)

remaining_suspicious

In [ ]:
second_rule_candidate = (
    (
        (deduplicated_df["collected_count"] >= 100_000)
        & (deduplicated_df["collection_like_ratio"] >= 100)
    )
    |
    (
        (deduplicated_df["comments_count"] >= 100_000)
        & (deduplicated_df["comment_like_ratio"] >= 100)
    )
)

second_rule_review = deduplicated_df.loc[
    second_rule_candidate,
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "collection_like_ratio",
        "comment_like_ratio",
        "domain"
    ]
].sort_values(
    "total_interactions",
    ascending=False
)

print("第二层候选记录数：", len(second_rule_review))
print(
    "涉及的不同note_id数量：",
    second_rule_review["note_id"].nunique()
)

second_rule_review.head(20)

## 阈值稍微变化，结论会不会完全不同。这叫“敏感性检查”，也是面试时很有价值的谈论点

In [ ]:
absolute_thresholds = [
    50_000,
    100_000,
    200_000
]

ratio_thresholds = [
    50,
    100,
    200
]

sensitivity_results = []

for absolute_threshold in absolute_thresholds:
    for ratio_threshold in ratio_thresholds:

        candidate_mask = (
            (
                (deduplicated_df["collected_count"] >= absolute_threshold)
                & (
                    deduplicated_df["collection_like_ratio"]
                    >= ratio_threshold
                )
            )
            |
            (
                (deduplicated_df["comments_count"] >= absolute_threshold)
                & (
                    deduplicated_df["comment_like_ratio"]
                    >= ratio_threshold
                )
            )
        )

        sensitivity_results.append(
            {
                "绝对量阈值": absolute_threshold,
                "比例阈值": ratio_threshold,
                "候选记录数": candidate_mask.sum()
            }
        )

sensitivity_df = pd.DataFrame(
    sensitivity_results
)

sensitivity_df

In [ ]:
loose_candidate_mask = (
    (
        (deduplicated_df["collected_count"] >= 50_000)
        & (deduplicated_df["collection_like_ratio"] >= 50)
    )
    |
    (
        (deduplicated_df["comments_count"] >= 50_000)
        & (deduplicated_df["comment_like_ratio"] >= 50)
    )
)

loose_candidates = deduplicated_df.loc[
    loose_candidate_mask,
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "collection_like_ratio",
        "comment_like_ratio",
        "local_time",
        "domain"
    ]
].sort_values(
    "total_interactions",
    ascending=False
)

loose_candidates

In [ ]:
high_collection_id = (
    loose_candidates.nlargest(
        1,
        "collection_like_ratio"
    )["note_id"].iloc[0]
)

print(
    "自动取得的note_id：",
    high_collection_id
)

high_collection_versions = df.loc[
    df["note_id"] == high_collection_id,
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "total_interactions",
        "collection_like_ratio",
        "comment_like_ratio",
        "candidate_anomaly",
        "local_time",
        "domain",
        "time"
    ]
].sort_values(
    "total_interactions"
)

high_collection_versions

In [ ]:
df["candidate_anomaly_v2"] = (
    df["candidate_anomaly"]
    |
    (
        (df["collected_count"] >= 100_000)
        & (df["collection_like_ratio"] >= 50)
    )
    |
    (
        (df["comments_count"] >= 100_000)
        & (df["comment_like_ratio"] >= 50)
    )
)

newly_flagged = df.loc[
    df["candidate_anomaly_v2"]
    & ~df["candidate_anomaly"],
    [
        "note_id",
        "liked_count",
        "collected_count",
        "comments_count",
        "collection_like_ratio",
        "comment_like_ratio",
        "candidate_anomaly",
        "candidate_anomaly_v2"
    ]
]

print(
    "旧版候选记录数：",
    df["candidate_anomaly"].sum()
)

print(
    "新版候选记录数：",
    df["candidate_anomaly_v2"].sum()
)

print(
    "新版涉及的不同note_id数：",
    df.loc[
        df["candidate_anomaly_v2"],
        "note_id"
    ].nunique()
)

print(
    "相对旧版新增记录数：",
    len(newly_flagged)
)

newly_flagged

## 重新生成第二版清洗表

In [ ]:
# 最终互动有效性标记
df["invalid_interaction"] = (
    df["candidate_anomaly_v2"]
)

# 第一步：排除互动数据无效的原始记录
valid_records_v2 = df.loc[
    ~df["invalid_interaction"]
].copy()

# 第二步：重新计算元数据完整度
valid_records_v2["metadata_completeness"] = (
    valid_records_v2[
        [
            "time",
            "domain",
            "user_id"
        ]
    ]
    .notna()
    .sum(axis=1)
)

# 第三步：同一作品优先保留总互动量高、
# 元数据更完整的正常版本
deduplicated_df_v2 = (
    valid_records_v2
    .sort_values(
        [
            "note_id",
            "total_interactions",
            "metadata_completeness"
        ],
        ascending=[
            True,
            False,
            False
        ],
        kind="mergesort"
    )
    .drop_duplicates(
        subset="note_id",
        keep="first"
    )
    .copy()
)

print(
    "排除无效互动后的原始记录数：",
    len(valid_records_v2)
)

print(
    "第二版去重表记录数：",
    len(deduplicated_df_v2)
)

print(
    "第二版仍重复的note_id数：",
    deduplicated_df_v2["note_id"].duplicated().sum()
)

print(
    "刚确认的两条记录剩余数量：",
    deduplicated_df_v2["note_id"]
    .isin(newly_flagged["note_id"])
    .sum()
)

In [ ]:
final_columns = [
    "note_id",
    "note_title",
    "time",
    "local_time",
    "desc",
    "liked_count",
    "collected_count",
    "comments_count",
    "domain",
    "user_id",
    "total_interactions"
]

# 用验证通过的第二版结果覆盖旧版clean_df
clean_df = (
    deduplicated_df_v2[
        final_columns
    ]
    .copy()
)

# 重新生成后续分析需要的派生字段
clean_df["publish_datetime"] = pd.to_datetime(
    clean_df["local_time"].astype(str),
    format="%Y%m%d%H",
    errors="coerce"
)

clean_df["domain_analysis"] = (
    clean_df["domain"].fillna("未知")
)

clean_df["has_user_id"] = (
    clean_df["user_id"].notna()
)

clean_df["publish_year"] = (
    clean_df["publish_datetime"].dt.year
)

clean_df["publish_year_month"] = (
    clean_df["publish_datetime"]
    .dt.to_period("M")
    .astype(str)
)

print(
    "最终分析表形状：",
    clean_df.shape
)

print(
    "重复note_id数量：",
    clean_df["note_id"].duplicated().sum()
)

print(
    "无法转换的发布时间数量：",
    clean_df["publish_datetime"].isna().sum()
)

In [ ]:
anomaly_ids = set(
    df.loc[df["candidate_anomaly_v2"],"note_id"]
)

valid_ids = set(
    valid_records_v2["note_id"].dropna()
)

recoverable_ids = anomaly_ids & valid_ids
only_anomaly_ids = anomaly_ids - valid_ids

print("原始记录数：", len(df))
print("原始唯一 note_id 数：", df["note_id"].nunique())
print("新版异常记录数：", int(df["candidate_anomaly_v2"].sum()))
print("新版异常涉及的 note_id 数：", len(anomaly_ids))
print("异常 ID 中有正常替代版本的数量：", len(recoverable_ids))
print("只有异常版本、没有正常替代的 ID 数：", len(only_anomaly_ids))
print("排除异常记录后剩余行数：", len(valid_records_v2))
print("有效记录去重减少的行数：", len(valid_records_v2) - len(deduplicated_df_v2))
print("最终记录数：", len(deduplicated_df_v2))
print("最终唯一 note_id 数：", deduplicated_df_v2["note_id"].nunique())

print(
    "记录数核对通过：",
    len(df) - len(valid_records_v2) == int(df["candidate_anomaly_v2"].sum())
)
print(
    "ID 数核对通过：",
    df["note_id"].nunique() - len(only_anomaly_ids)
    == deduplicated_df_v2["note_id"].nunique()
)


## 第二版清洗结果核对

原始数据共有 91,517 条记录，对应 60,161 个不同的 `note_id`。

新版异常规则标记了 230 条记录，涉及 217 个不同的 `note_id`。其中15个 ID 仍有正常记录可以保留，202个 ID 只有异常版本。排除异常记录后剩余91,287条记录，再删除31,328条重复记录，最终得到59,959条记录和59,959个不同的 ID。记录数与 ID 数核对结果均通过。

同一 `note_id` 存在多个正常版本时，当前规则优先保留总互动量更高的记录；总互动量相同时，优先保留元数据更完整的记录。由于数据没有可靠的抓取时间，互动量更高只能作为可复现的去重规则，不能证明该记录一定是最新或最准确的版本。

In [ ]:
processed_path = "../data/processed/notes_clean_v2.csv"

clean_df.to_csv(
    processed_path,
    index=False,
    encoding="utf-8-sig"
)

print("导出行数：", len(clean_df))
print("导出列数：", len(clean_df.columns))
print("保存位置：", processed_path)

In [ ]:
reloaded_df = pd.read_csv(
    processed_path,
    encoding="utf-8-sig"
)

print("回读后的形状：", reloaded_df.shape)

print(
    "行列数一致：",
    reloaded_df.shape == clean_df.shape
)

print(
    "字段顺序一致：",
    list(reloaded_df.columns) == list(clean_df.columns)
)

print(
    "回读后的唯一 note_id 数：",
    reloaded_df["note_id"].nunique()
)

print(
    "回读后的重复 note_id 数：",
    reloaded_df["note_id"].duplicated().sum()
)

print(
    "回读后的缺失 note_id 数：",
    reloaded_df["note_id"].isna().sum()
)